<a href="https://colab.research.google.com/github/contreras-juan/Material-Ciencia-de-Datos/blob/main/Deep_Learning/Ejercicios/02_Taller_Convoluciones_y_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<h1 style="color: #FECB05; text-align: center;">Taller práctico: Convoluciones y CNN</h1>


<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)


---


<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller integra los temas de:

1. Introducción a las convoluciones (`05_Convoluciones.ipynb`)
2. Redes neuronales convolucionales (`06_CNN.ipynb`)

**Cómo trabajar:**

- Completa las celdas marcadas con `# Escribe tu código aquí`.
- Responde las preguntas de reflexión.
- Usa NumPy, SciPy, Matplotlib, scikit-learn y TensorFlow/Keras.
- Prioriza entender el efecto de kernels, padding, stride, pooling y la arquitectura CNN.

**Tiempo sugerido:** 2.5–3.5 horas.


<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [Ejercicio 1. Convolución 1D](#ejercicio-1)
- [Ejercicio 2. Convolución 2D y tamaño de salida](#ejercicio-2)
- [Ejercicio 3. Filtros sobre imágenes](#ejercicio-3)
- [Ejercicio 4. Padding, stride y pooling](#ejercicio-4)
- [Ejercicio 5. Parámetros de una capa Conv2D](#ejercicio-5)
- [Ejercicio 6. CNN para clasificar dígitos](#ejercicio-6)
- [Ejercicio 7. Regularización y aumentación](#ejercicio-7)
- [Ejercicio integrador](#ejercicio-integrador)


---


<h2 style="color: #007ACC;">Instalación e importaciones</h2>

Ejecuta primero la celda de instalación y luego la de importaciones.


In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scipy Pillow scikit-learn tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

from scipy.signal import convolve2d

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import L2

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Convolución 1D</h2>

**Objetivo:** calcular convoluciones discretas a mano y con NumPy.


<h3 style="color: #003366;">1.1 Cálculo directo</h3>

Sean

$$f = [1,\ 2,\ 3,\ 4],\qquad g = [1,\ 0,\ -1].$$

1. Calcula $f * g$ en modo `full` (longitud $|f|+|g|-1$).
2. Verifica el resultado con `np.convolve`.
3. Interpreta qué hace este kernel $g$ sobre una señal 1D.


In [ ]:
# Escribe tu código aquí
f = np.array([1, 2, 3, 4])
g = np.array([1, 0, -1])

# 1) Resultado esperado a mano (puedes dejarlo comentado)
# full = [...]

# 2) Verificación con NumPy
# print(np.convolve(...))


<h3 style="color: #003366;">1.2 Convolución de pmfs</h3>

Sea $p = [1/6,\ldots,1/6]$ la pmf de un dado justo. Calcula $C = p * p$ y grafica $P(\text{suma}=s)$ para $s=2,\ldots,12$.


In [ ]:
# Escribe tu código aquí


---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Convolución 2D y tamaño de salida</h2>

**Objetivo:** practicar convolución 2D y la fórmula de tamaño de salida.


Usa

$$
X=\begin{bmatrix}
1&2&3&0\\
0&1&2&3\\
3&0&1&2\\
2&3&0&1
\end{bmatrix},\qquad
K=\begin{bmatrix}
1&0&-1\\
1&0&-1\\
1&0&-1
\end{bmatrix}.
$$

1. Calcula `convolve2d(X, K, mode="valid")` y `mode="same"`.
2. Con $H_{in}=W_{in}=4$, $F=3$, $S=1$:
   - sin padding ($P=0$), ¿cuál es $H_{out}$?
   - con padding $P=1$, ¿cuál es $H_{out}$?
3. Comprueba que tus tamaños coinciden con las matrices obtenidas.


In [ ]:
# Escribe tu código aquí
X = np.array([
    [1, 2, 3, 0],
    [0, 1, 2, 3],
    [3, 0, 1, 2],
    [2, 3, 0, 1],
], dtype=float)

K = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
], dtype=float)


**Pregunta:** en CNN de Keras, ¿la operación de `Conv2D` corresponde exactamente a la convolución matemática de SciPy o a correlación cruzada? Justifica en una frase.


In [ ]:
# Respuesta:


---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Filtros sobre imágenes</h2>

**Objetivo:** aplicar kernels clásicos (media, gaussiano, laplaciano) a una imagen RGB.


1. Carga `../img/demo_convoluciones.png` (desde `Ejercicios/`; si abres desde otra ruta o Colab, ajusta la ruta o genera una imagen sintética).
2. Implementa una función `aplicar_kernel(img, kernel)` que convolucione canal por canal (`mode="same"`).
3. Aplica:
   - desenfoque promedio $3\times 3$
   - gaussiano $5\times 5$
   - laplaciano (visualiza con valor absoluto + reescalado a uint8)
4. Muestra original vs cada resultado.


In [ ]:
# Escribe tu código aquí
def gen_gaussian_kernel(N, sigma):
    pass

def aplicar_kernel(img, kernel):
    pass


---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Padding, stride y pooling</h2>

**Objetivo:** implementar stride y pooling, y relacionarlos con reducción espacial.


Sobre la matriz

$$
M=\begin{bmatrix}
1&3&2&4\\
5&6&1&2\\
7&8&3&0\\
2&1&9&5
\end{bmatrix}
$$

1. Implementa `pool2d(M, size=2, stride=2, mode="max"|"avg")`.
2. Implementa una convolución 2D `valid` con stride configurable (puedes usar un kernel promedio $2\times 2$).
3. Compara shapes de:
   - max-pool $2\times 2$, stride 2
   - conv $2\times 2$, stride 2
4. Explica en una frase cuándo preferirías pooling frente a convolución con stride.


In [ ]:
# Escribe tu código aquí
M = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [7, 8, 3, 0],
    [2, 1, 9, 5],
], dtype=float)


---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Parámetros de una capa Conv2D</h2>

**Objetivo:** contar parámetros aprendibles de capas convolucionales.


Usa la fórmula

$$\text{params} = (F^{2}\cdot C_{in} + 1)\cdot K$$

1. Calcula a mano los parámetros de:
   - `Conv2D(32, 3x3)` con entrada RGB ($C_{in}=3$)
   - `Conv2D(64, 3x3)` que recibe la salida de la capa anterior ($C_{in}=32$)
2. Construye esas dos capas en Keras (puedes usar un `Sequential` pequeño) y verifica con `model.summary()` o `layer.count_params()`.


In [ ]:
# Escribe tu código aquí


---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. CNN para clasificar dígitos</h2>

**Objetivo:** entrenar una CNN básica de punta a punta.


Usa `sklearn.datasets.load_digits` (imágenes $8\times 8$):

1. Normaliza a $[0,1]$ y da forma `(N, 8, 8, 1)`.
2. Codifica etiquetas con one-hot.
3. Separa train/test (80/20, estratificado si es posible).
4. Define una CNN con al menos:
   - `Conv2D` + `ReLU`
   - `MaxPooling2D`
   - `Flatten`
   - `Dense` oculta
   - salida `softmax` (10 clases)
5. Entrena con `validation_split=0.2` y evalúa en test.
6. Grafica curvas de accuracy/loss y una matriz de confusión.


In [ ]:
# Escribe tu código aquí
digits = load_digits()
X = digits.images
y = digits.target


---

<a id="ejercicio-7"></a>
<h2 style="color: #007ACC;">Ejercicio 7. Regularización y aumentación</h2>

**Objetivo:** aplicar data augmentation, BatchNormalization y Dropout en una CNN.


Sobre el mismo problema de dígitos (o MNIST si prefieres):

1. Configura un `ImageDataGenerator` de entrenamiento con al menos rotación y traslación (y `rescale` si aplica).
2. Para validación/test usa solo rescalado (sin augmentación).
3. Entrena una CNN con:
   - `BatchNormalization`
   - `Dropout`
   - opcional: `kernel_regularizer=L2(...)`
   - callbacks: `EarlyStopping` + `ReduceLROnPlateau`
4. Compara el resultado con el modelo del ejercicio 6 (test accuracy y forma de las curvas).

**Nota:** como `load_digits` es pequeño, la augmentación puede ayudar poco o incluso añadir ruido; lo importante es implementar el flujo correctamente y observar el efecto.


In [ ]:
# Escribe tu código aquí


---

<a id="ejercicio-integrador"></a>
<h2 style="color: #007ACC;">Ejercicio integrador</h2>

**Objetivo:** comparar un MLP denso frente a una CNN en el mismo problema de imágenes.


Con dígitos (`8x8`):

1. Entrena un **MLP**: aplana cada imagen a vector de 64 y usa 1–2 capas `Dense`.
2. Entrena una **CNN** (puedes reutilizar la del ejercicio 6 o 7).
3. Usa el mismo split train/test y un presupuesto similar de épocas + `EarlyStopping`.
4. Reporta en una tablita:
   - exactitud en test
   - número de parámetros
   - épocas efectivas
5. Conclusión: ¿qué modelo generaliza mejor y por qué tiene sentido en datos con estructura espacial?


In [ ]:
# Escribe tu código aquí
# Experimento MLP vs CNN
